# Unsway — Phase 6D train-only directions and validation

This notebook rebuilds or restores every prerequisite, constructs directions from the **train split only**, evaluates doses on **validation only**, and keeps the replacement test pressure/control prompts unopened unless a candidate passes the frozen guardrails.

In [ ]:
from pathlib import Path

repo = Path("/content/Unsway")
if (repo / ".git").is_dir():
    !git -C /content/Unsway pull --ff-only
else:
    !git clone https://github.com/idris404/Unsway.git /content/Unsway
%cd /content/Unsway
!pip install -q -e '.[dev]'

In [ ]:
import torch

assert torch.cuda.is_available(), "Select a GPU runtime before continuing."
print("CUDA device:", torch.cuda.get_device_name(0))

## Rebuild every prerequisite in this runtime

The workflow is self-contained and does not request access to Google Drive.

In [ ]:
print("Prerequisites will be rebuilt from checksum-pinned sources.")

## Rebuild frozen datasets and behavior artifacts

In [ ]:
import json
import subprocess

def run(command, expected=(0,)):
    print("+", " ".join(command))
    result = subprocess.run(command)
    assert result.returncode in expected, (result.returncode, command)

run(["python", "-m", "unsway.cli.phase1", "--config", "configs/phase1.yaml"])
run(["python", "-m", "unsway.cli.phase6", "--config", "configs/phase6.yaml", "--stage", "data"])
run(["python", "-m", "unsway.cli.phase6", "--config", "configs/phase6c.yaml", "--stage", "data"])

gate_report = Path("reports/phase6c_baseline.json")
gate_predictions = Path("data/processed/phase6c_test_initial_predictions.jsonl")
if not (gate_report.is_file() and gate_predictions.is_file()):
    run(["python", "-m", "unsway.cli.phase6", "--config", "configs/phase6c.yaml", "--stage", "baseline"])
gate = json.loads(gate_report.read_text())
assert gate["status"] == "ready_for_frozen_test"
assert gate["test_initial_only"]["metrics"]["overall"]["initial_correct_trials"] == 537
assert gate["test_initial_only"]["pressure_scored"] is False
assert gate["test_initial_only"]["control_scored"] is False

v2_report = Path("reports/phase6_baseline.json")
v2_predictions = Path("data/processed/phase6_train_validation_predictions.jsonl")
if not (v2_report.is_file() and v2_predictions.is_file()):
    run(["python", "-m", "unsway.cli.phase6", "--config", "configs/phase6.yaml", "--stage", "baseline"], expected=(1,))
retired = json.loads(v2_report.read_text())
assert retired["status"] == "insufficient_test_eligibility"
assert retired["test_initial_only"]["metrics"]["overall"]["initial_correct_trials"] == 299
print("Behavior artifacts verified; replacement pressure/control test remains unopened.")

## Rebuild train/validation activations and the legacy Phase 3 SAE

In [ ]:
run(["python", "-m", "unsway.cli.phase6", "--config", "configs/phase6.yaml", "--stage", "extract", "--eligibility-config", "configs/phase6c.yaml"])

legacy_sae = Path("data/processed/phase3/sae.safetensors")
legacy_training = Path("reports/phase3_training.json")
if not (legacy_sae.is_file() and legacy_training.is_file()):
    run(["python", "-m", "unsway.cli.phase2", "--config", "configs/phase2.yaml"])
    run(["python", "-m", "unsway.cli.phase3", "--config", "configs/phase3.yaml", "--stage", "extract"])
    run(["python", "-m", "unsway.cli.phase3", "--config", "configs/phase3.yaml", "--stage", "train"])
from unsway.data.source import sha256_file
assert sha256_file(legacy_sae) == "c01496e787d82b529547f586cd51a87319a2a5d8403b5ebe544985cbf6432798"
print("Legacy SAE checksum verified.")

## Construct train-only directions and finish validation

This is the decision point. The command does not score replacement test pressure/control prompts.

In [ ]:
run(["python", "-m", "unsway.cli.phase6", "--config", "configs/phase6.yaml", "--stage", "phase6d", "--methods-config", "configs/phase6d.yaml"])

In [ ]:
validation = json.loads(Path("reports/phase6d_validation.json").read_text())
best_by_intervention = {}
for row in validation["conditions"]:
    if row["family"] == "matched_random":
        continue
    previous = best_by_intervention.get(row["intervention"])
    if previous is None or row["delta"]["pressure_effect"] < previous["delta"]["pressure_effect"]:
        best_by_intervention[row["intervention"]] = row
summary = {
    "status": validation["status"],
    "selected_confirmatory_candidate": validation["selected_confirmatory_candidate"],
    "best_by_intervention": {
        name: {
            "strength": row["strength"],
            "delta": row["delta"],
            "source_pressure_effect_deltas": row["source_pressure_effect_deltas"],
        } for name, row in best_by_intervention.items()
    },
    "test_prompts_scored": validation["test_prompts_scored"],
}
print(json.dumps(summary, indent=2))

## Download the complete Phase 6D bundle

In [ ]:
import zipfile
from google.colab import files

artifacts = [
    Path("reports/phase6d_methods.json"),
    Path("reports/phase6d_directions.json"),
    Path("reports/phase6d_validation.json"),
    Path("data/processed/phase6/phase6d_directions.safetensors"),
    Path("data/processed/phase6/phase6d_validation_predictions.jsonl"),
]
bundle = Path("/content/phase6d_bundle.zip")
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for source in artifacts:
        archive.write(source, arcname=source.name)
print("Bundle ready:", bundle)
files.download(str(bundle))